# FFAI Message Stack: Live LLM Trace

This notebook uses the **real Mistral Small API** via `FFLiteLLMClient` to show
exactly what messages are sent to the LLM on each of 10 turns, across three
scenarios:

- **Scenario A**: Plain calls — the client's `conversation_history` grows every turn
- **Scenario B**: `history=["prev"]` — only the referenced turn is injected as XML
- **Scenario C**: `{{name.response}}` — the prior response is interpolated inline

Each turn prints the full `messages` array that `litellm.completion()` receives.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from ffai.Clients.FFLiteLLMClient import FFLiteLLMClient  # noqa: E402
from ffai.FFAI import FFAI  # noqa: E402

In [ ]:
captured_messages = []


def make_traced_client():
    client = FFLiteLLMClient(
        model_string='mistral/mistral-small-2503',
        temperature=0.3,
        max_tokens=256,
        system_instructions='You are a helpful assistant. Be very brief.',
    )

    original_prepare = client._prepare_generate_params

    def traced_prepare(prompt, model=None, system_instructions=None,
                        temperature=None, max_tokens=None, **kwargs):
        params, model_str = original_prepare(
            prompt, model, system_instructions, temperature, max_tokens, **kwargs
        )
        captured_messages.append({
            'turn': len(captured_messages) + 1,
            'model': model_str,
            'messages': [dict(m) for m in params['messages']],
        })
        return params, model_str

    client._prepare_generate_params = traced_prepare
    return client


def print_turn(data, max_len=200):
    msgs = data['messages']
    print(f'Turn {data["turn"]} | {data["model"]} | {len(msgs)} messages')
    print()
    for m in msgs:
        role = m['role'].upper()
        content = m['content']
        if len(content) > max_len:
            content = content[:max_len] + f' ... [{len(m["content"])} chars]'
        print(f'  [{role}]')
        for line in content.split('\n'):
            print(f'    {line}')
        print()


print('Ready')

---

## Scenario A: 10 Plain Turns (No History References)

No `history`, no `{{}}`. The client's internal `conversation_history` grows
naturally — every prior user/assistant pair is sent in the `messages` array.

In [ ]:
captured_messages.clear()
client_a = make_traced_client()
ffai_a = FFAI(client_a)

for i in range(1, 11):
    result = ffai_a.generate_response(
        f'Turn {i}: Tell me one fact about the number {i}.',
        prompt_name=f'turn_{i}',
    )
    print(f'Turn {i}: {result.response[:80]}')

captured_a = list(captured_messages)
print(f'\nCaptured {len(captured_a)} turns')
print(f'Client conversation_history: {len(client_a.conversation_history)} messages')

In [ ]:
for turn_num in [1, 2, 5, 10]:
    print(f'===== SCENARIO A: Turn {turn_num} =====')
    print_turn(captured_a[turn_num - 1], max_len=120)
    print()

In [ ]:
print('Message growth (Scenario A):')
print()
for t in captured_a:
    n = len(t['messages'])
    bar = '#' * n
    print(f'  Turn {t["turn"]:2d}: {n:2d} messages  {bar}')
print()
print('Formula: 1 system + 2*(turn-1) prior + 1 user = 2*turn')

---

## Scenario B: 10 Turns With `history=["prev"]`

Each turn references the **previous turn** via `history=["turn_N"]`. FFAI:

1. **Saves** the client's `conversation_history` and **clears** it
2. Injects the referenced turn as `<conversation_history>` XML in the user message
3. After the call, **merges** saved + new messages back

The LLM always sees exactly 2 messages: system + user (with XML context).

In [ ]:
captured_messages.clear()
client_b = make_traced_client()
ffai_b = FFAI(client_b)

result = ffai_b.generate_response('My favorite color is blue.', prompt_name='turn_1')
print(f'Turn 1: {result.response[:80]}')

for i in range(2, 11):
    result = ffai_b.generate_response(
        f'Turn {i}: What was my favorite color? Say it in one word.',
        prompt_name=f'turn_{i}',
        history=[f'turn_{i-1}'],
    )
    print(f'Turn {i}: {result.response[:80]}')

captured_b = list(captured_messages)
print(f'\nCaptured {len(captured_b)} turns')

In [ ]:
for turn_num in [1, 2, 3, 10]:
    ref = 'none' if turn_num == 1 else f'turn_{turn_num-1}'
    print(f'===== SCENARIO B: Turn {turn_num} (history=["{ref}"]) =====')
    print_turn(captured_b[turn_num - 1], max_len=300)
    print()

---

## Scenario C: Interpolation with `{{name.response}}`

Each turn interpolates the prior response **inline** into the prompt.
No XML block — the prior response text is substituted directly.
The LLM always sees exactly 2 messages: system + user.

In [ ]:
captured_messages.clear()
client_c = make_traced_client()
ffai_c = FFAI(client_c)

result = ffai_c.generate_response('My favorite color is blue.', prompt_name='turn_1')
print(f'Turn 1: {result.response[:80]}')

for i in range(2, 11):
    prompt = 'You previously said: {{turn_' + str(i-1) + '.response}}. Repeat that color in one word.'
    result = ffai_c.generate_response(prompt, prompt_name=f'turn_{i}')
    print(f'Turn {i}: {result.response[:80]}')

captured_c = list(captured_messages)
print(f'\nCaptured {len(captured_c)} turns')

In [ ]:
for turn_num in [1, 2, 3, 10]:
    print(f'===== SCENARIO C: Turn {turn_num} =====')
    print_turn(captured_c[turn_num - 1], max_len=300)
    print()

---

## Complete 10-Turn Comparison Table

All three scenarios side by side, showing whether the full message stack
is sent and what extra context is included on each turn.

In [ ]:
results = {}

# --- A: Plain ---
captured_messages.clear()
_c = make_traced_client()
_f = FFAI(_c)
for i in range(1, 11):
    _f.generate_response(f'Turn {i}: one fact about {i}.', prompt_name=f'turn_{i}')
results['A'] = list(captured_messages)

# --- B: history ---
captured_messages.clear()
_c = make_traced_client()
_f = FFAI(_c)
_f.generate_response('My favorite color is blue.', prompt_name='turn_1')
for i in range(2, 11):
    _f.generate_response(
        f'Turn {i}: repeat my favorite color.',
        prompt_name=f'turn_{i}',
        history=[f'turn_{i-1}'],
    )
results['B'] = list(captured_messages)

# --- C: interpolation ---
captured_messages.clear()
_c = make_traced_client()
_f = FFAI(_c)
_f.generate_response('My favorite color is blue.', prompt_name='turn_1')
for i in range(2, 11):
    _p = 'You said: {{turn_' + str(i-1) + '.response}} Repeat that color.'
    _f.generate_response(_p, prompt_name=f'turn_{i}')
results['C'] = list(captured_messages)


def describe(t):
    n = len(t['messages'])
    user = t['messages'][-1]['content']
    full = n > 2
    extras = []
    if '<conversation_history>' in user:
        extras.append(user.count('<interaction '))
    return full, n, extras


print(f'{"#":>2}  {"--- A: Plain ---":^32}  {"--- B: history ---":^32}  {"--- C: interp ---":^32}')
print(f'{"":>2}  {"Stk Msgs Extra":^32}  {"Stk Msgs Extra":^32}  {"Stk Msgs Extra":^32}')
print(f'{"":>2}  {"-" * 32:^32}  {"-" * 32:^32}  {"-" * 32:^32}')

for i in range(10):
    a_fs, a_n, a_e = describe(results['A'][i])
    b_fs, b_n, b_e = describe(results['B'][i])
    c_fs, c_n, c_e = describe(results['C'][i])
    b_desc = f'{b_e[0]} XML block(s)' if b_e else 'none'
    c_desc = 'interpolated' if not c_fs else 'none'
    a_str = f'{"Y" if a_fs else "-":>1} {a_n:>3}  {"prior msgs" if a_fs else "none":^21}'
    b_str = f'{"Y" if b_fs else "-":>1} {b_n:>3}  {b_desc:^21}'
    c_str = f'{"Y" if c_fs else "-":>1} {c_n:>3}  {c_desc:^21}'
    print(f'{i+1:>2}  {a_str}  {b_str}  {c_str}')

print()
print('Stack=Yes: client conversation_history sent (grows each turn)')
print('Stack=No:  client history suspended; context embedded in user message')

---

## Key Takeaways

| Mechanism | Trigger | Messages sent | Where context goes |
|-----------|---------|---------------|--------------------|
| **Plain** | No `history`, no `{{}}` | `2n` (grows each turn) | In the `messages` array as prior user/assistant pairs |
| **`history=["X"]`** | Explicit list | Always 2 | `<conversation_history>` XML inside the user message |
| **`{{X.response}}`** | Interpolation | Always 2 | Inline text substitution inside the user prompt |

**The suspend/restore mechanism** is what keeps Scenarios B and C at 2 messages:
the client's conversation history is saved and cleared before each call,
then restored afterward. The LLM never sees the growing history — only
the explicitly requested context.